#### Changing the directory

In [3]:
%pwd

'd:\\Office\\ODCRU\\02_During_Prep_Prog\\ODCRU-AI-Projects\\experiment'

In [4]:
import os
os.chdir("../")
%pwd

'd:\\Office\\ODCRU\\02_During_Prep_Prog\\ODCRU-AI-Projects'

#### Use of ConfigBox

In [5]:
dicta = {"a":1,"b":2}
dicta["a"]

1

In [ ]:
# dicta.a   ### AttributeError as dict does not support attribute-style access 
            ### so we need to use a custom class or a library like box

AttributeError: 'dict' object has no attribute 'a'

In [6]:
from box import ConfigBox

dicta = ConfigBox(dicta)
dicta.a

1

#### Reading the YAML File

In [ ]:
import re
def resolve_config(content: dict):
    '''
    This function resolves the placeholders in the configuration dictionary.
    Placeholders are in the format ${key1.key2} and will be replaced with the
    corresponding value from the config dictionary.
    Args:
        config (dict): The configuration dictionary with potential placeholders.
    Errors:
        Exception: Raises an exception if a placeholder cannot be resolved.
    Returns:
        dict: The configuration dictionary with placeholders resolved.
        
    '''
    pattern = re.compile(r"\$\{([^}^{]+)\}")

    def replacer(match):
        try:
            keys = match.group(1).split(".")
            value = content
            for k in keys:
                value = value[k]
            return str(value)
        except Exception as e:
            raise e

    def walk(d):
        for k, v in d.items():
            if isinstance(v, dict):
                walk(v)
            elif isinstance(v, str):
                d[k] = pattern.sub(replacer, v)

    walk(content)
    return content

In [25]:
import yaml

def read_yaml(path_to_yaml: str) -> ConfigBox:
    """Reads a yaml file and returns a ConfigBox object.
    Args:
        path_to_yaml (str): path like input
    Errors:
        exception: Empty file or any other exception while reading yaml file
    Returns:
        ConfigBox: A ConfigBox object containing the loaded YAML data
    """
    try:
        with open(path_to_yaml) as yaml_file:
            content = yaml.safe_load(yaml_file)
            content = resolve_config(content)
            return ConfigBox(content)
    except Exception as e:
        raise e  
    
    

In [29]:
path_to_yaml = "config/config.yaml"
config = read_yaml(path_to_yaml)
config

ConfigBox({'artifacts_root': 'artifacts', 'naive_baysian': {'save_model_path': 'artifacts/model/naive_baysian.pkl'}, 'data_ingestion': {'data_root': 'data', 'raw_data_dir': 'data', 'raw_data_file': 'sentiment_analysis.csv', 'raw_data_file_path': 'data/sentiment_analysis.csv'}})

In [30]:

config.naive_baysian.save_model_path

'artifacts/model/naive_baysian.pkl'

In [31]:
config.data_ingestion.raw_data_file_path

'data/sentiment_analysis.csv'

#### Data Ingestion

In [40]:
import pandas as pd
from pandas import DataFrame

def read_csv(path: str) -> DataFrame:
    """Reads a CSV file and returns a DataFrame.
    Args:
        path (str): path like input
    Errors:
        exception: File not found or any other exception while reading CSV file
    Returns:
        pd.DataFrame: DataFrame containing the CSV data
    """
    try:
        df = pd.read_csv(path)
        df.columns = df.columns.str.strip()
        df = df.dropna(subset=['text', 'sentiment'])
        return df
    except Exception as e:
        raise e

In [41]:
df = read_csv(path=config.data_ingestion.raw_data_file_path)
df.head()

,Year,Month,Day,Time of Tweet,text,sentiment,Platform
0,2018,8,18,morning,What a great day!!! Looks like dream.,positive,Twitter
1,2018,8,18,noon,"I feel sorry, I miss you here in the sea beach",positive,Facebook
2,2017,8,18,night,Don't angry me,negative,Facebook
3,2022,6,8,morning,We attend in the class just for listening teac...,negative,Facebook
4,2022,6,8,noon,"Those who want to go, let them go",negative,Instagram
